# kami-oracle starter

Minimal Colab notebook for querying the live oracle over HTTPS.

Setup once:
1. Add a Colab secret named `KAMI_ORACLE_TOKEN` (Tools → Secrets).
2. Set its value to the bearer token from the oracle VM's `.env` (`KAMI_ORACLE_API_TOKEN`).
3. Toggle “Notebook access” so this notebook can read it.

Then run the cells top-to-bottom. Examples query the rolling 7-day window.

In [ ]:
!pip install -q requests pandas

In [ ]:
from google.colab import userdata

ORACLE_URL = "https://136-112-224-147.sslip.io"
TOKEN = userdata.get("KAMI_ORACLE_TOKEN")

In [ ]:
import requests, pandas as pd

def sql(q, limit=1000):
    r = requests.post(
        f"{ORACLE_URL}/sql",
        headers={"Authorization": f"Bearer {TOKEN}"},
        json={"q": q, "limit": limit},
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data["rows"], columns=data["columns"])
    if data.get("truncated"):
        print(f"⚠️ truncated at {data['row_count']} rows")
    return df

# Sanity check.
requests.get(f"{ORACLE_URL}/health", timeout=10).json()

## Example 1 — top 20 harvesters (last 7 days)

Counts `harvest_start` actions per kami. A high count means the kami is being cycled aggressively (harvest → collect → stop → harvest again).

In [ ]:
sql("""
    SELECT kami_id, COUNT(*) AS harvest_starts
    FROM kami_action
    WHERE action_type = 'harvest_start'
      AND block_timestamp > now() - INTERVAL 7 DAY
    GROUP BY kami_id
    ORDER BY 2 DESC
    LIMIT 20
""")

## Example 2 — action-type histogram (last 7 days)

Cross-checks against the pre-baked `/actions/types` endpoint — the totals should match.

In [ ]:
df_sql = sql("""
    SELECT action_type, COUNT(*) AS n
    FROM kami_action
    WHERE block_timestamp > now() - INTERVAL 7 DAY
    GROUP BY 1
    ORDER BY 2 DESC
""")

rest_resp = requests.get(
    f"{ORACLE_URL}/actions/types?since_days=7",
    headers={"Authorization": f"Bearer {TOKEN}"},
    timeout=10,
).json()
df_rest = pd.DataFrame(rest_resp["by_type"]).set_index("action_type")["count"]

compare = df_sql.set_index("action_type").join(df_rest.rename("rest"))
compare["match"] = compare["n"] == compare["rest"]
compare

## Example 3 — bpeon operator timeline (last 7 days)

Pulls the founder's bpeon operator activity — the canonical reference account from Stage 1 validation. Useful as a smoke test that the data still looks sane.

In [ ]:
sql("""
    SELECT block_timestamp, action_type, kami_id, node_id, amount
    FROM kami_action
    WHERE from_addr = '0x86aDb8f741E945486Ce2B0560D9f643838FAcEC2'
      AND block_timestamp > now() - INTERVAL 7 DAY
    ORDER BY block_timestamp DESC
""", limit=2000)